# Shinhan Card 파일 불러오기

## 0. 초기 세팅
각종 라이브러리를 불러오고 초기값 등을 세팅합니다.

In [1]:
#!uv add xlrd
#!uv add lxml

In [21]:
from pathlib import Path
import pandas as pd

In [22]:
from ledgerly.expenditure import (shinhan_file_config
                                  , map_shinhan_card_df_to_expenditure
                                  , preprocess_shinhan_data
                                  , map_category
                                  , insert_expenditure_data
                                  , fetch_expenditure_data)


## 1. 데이터 파일 조회 및 전처리

In [23]:
sh_ori_df = pd.read_excel(shinhan_file_config["shinhan_xls"])
len(sh_ori_df)

8

In [24]:
sh_df = preprocess_shinhan_data(sh_ori_df)
sh_df.head()

,거래일,카드구분,이용카드,가맹점명,승인번호,금액,매입구분,이용구분,거래통화,해외이용금액,취소상태
0,2026-01-24 11:08:00,신용,본인204*,홈플러스㈜,25863165,68210,결제확정,일시불,NaN,NaN,NaN
1,2026-01-16 21:55:00,신용,본인204*,홈플러스㈜,31299218,50240,결제확정,일시불,NaN,NaN,NaN
2,2026-01-16 12:35:00,신용,본인204*,(주)아트박스 강서 홈플러스점,23635058,2000,결제확정,일시불,NaN,NaN,NaN
3,2026-01-07 16:48:00,신용,본인204*,새서울약국,8159190,5700,결제확정,일시불,NaN,NaN,NaN
4,2026-01-07 16:44:00,신용,본인204*,코아이비인후과,8112630,10800,결제확정,일시불,NaN,NaN,NaN


## 2. DB 데이터로 매핑

In [25]:
expenditure_df = map_shinhan_card_df_to_expenditure(sh_df)
expenditure_df.head()

,used_at,payment_type,payment_provider,merchant_name,installment_type,amount,remaining_amount,category,memo,source_uid
0,2026-01-24 11:08:00,card,shinhan,홈플러스㈜,single,68210,0,unknown,None,card_shinhan_25863165
1,2026-01-16 21:55:00,card,shinhan,홈플러스㈜,single,50240,0,unknown,None,card_shinhan_31299218
2,2026-01-16 12:35:00,card,shinhan,(주)아트박스 강서 홈플러스점,single,2000,0,unknown,None,card_shinhan_23635058
3,2026-01-07 16:48:00,card,shinhan,새서울약국,single,5700,0,unknown,None,card_shinhan_8159190
4,2026-01-07 16:44:00,card,shinhan,코아이비인후과,single,10800,0,unknown,None,card_shinhan_8112630


In [26]:
expenditure_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 10 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   used_at           7 non-null      datetime64[ns]
 1   payment_type      7 non-null      object        
 2   payment_provider  7 non-null      object        
 3   merchant_name     7 non-null      string        
 4   installment_type  7 non-null      object        
 5   amount            7 non-null      int64         
 6   remaining_amount  7 non-null      int64         
 7   category          7 non-null      object        
 8   memo              0 non-null      object        
 9   source_uid        7 non-null      object        
dtypes: datetime64[ns](1), int64(2), object(6), string(1)
memory usage: 692.0+ bytes


## 3. 지출 분류

In [27]:
expenditure_df["category"] = expenditure_df["merchant_name"].apply(map_category)
expenditure_df["category"] = expenditure_df["category"].fillna("미분류")

In [28]:
expenditure_df[["merchant_name", "category"]].head(20)

,merchant_name,category
0,홈플러스㈜,식비
1,홈플러스㈜,식비
2,(주)아트박스 강서 홈플러스점,식비
3,새서울약국,의료
4,코아이비인후과,기타
5,강서홈약국,의료
6,홈플러스㈜,식비


## 4. DB 저장

In [29]:
# insert_expenditure_data(expenditure_df)
# print(f"{len(expenditure_df)}개의 데이터가 성공적으로 저장되었습니다.")

In [30]:
fetched_db_df = fetch_expenditure_data()

In [31]:
fetched_db_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14 entries, 0 to 13
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                14 non-null     int64 
 1   used_at           14 non-null     object
 2   payment_type      14 non-null     object
 3   payment_provider  14 non-null     object
 4   merchant_name     14 non-null     object
 5   installment_type  14 non-null     object
 6   amount            14 non-null     int64 
 7   remaining_amount  14 non-null     int64 
 8   category          14 non-null     object
 9   memo              0 non-null      object
 10  source_uid        14 non-null     object
 11  created_at        14 non-null     object
dtypes: int64(3), object(9)
memory usage: 1.4+ KB
